# AORUS MASTER 16 AM6H Spec RAG — T4 benchmark

Everything in the repository runs on Apple Silicon during development, but Metal
uses unified memory and therefore cannot demonstrate the assignment's **4GB VRAM**
limit. This notebook exists to produce the numbers the README reports:

1. **VRAM evidence** — what the running system actually occupies on a discrete GPU.
2. **TTFT and TPS** on that GPU.
3. The retrieval and generation evaluations, reproduced end to end from a clean clone.

Runtime → Change runtime type → **T4 GPU** before running anything.

## 1. Confirm the GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


## 2. Clone the repository

In [2]:
!git clone --depth 1 https://github.com/maxxyhc/Gigabyte.git /content/repo
%cd /content/repo
!ls

Cloning into '/content/repo'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 31 (delta 0), reused 30 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 215.50 KiB | 1.03 MiB/s, done.
/content/repo
AGENTS.md  data  eval  notebooks  pyproject.toml  README.md  src  uv.lock


## 3. Environment via `uv`

`uv sync --frozen` installs exactly the versions in `uv.lock`, so this is also
the check that the committed lockfile reproduces the development environment.

In [3]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

!uv sync --frozen

downloading uv 0.12.5 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using CPython 3.12.14
Creating virtual environment at: .venv
Prepared 64 packages in 48.60s
Installed 64 packages in 588ms
 + annotated-doc==0.0.5
 + anyio==4.14.2
 + beautifulsoup4==4.15.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.1
 + click==8.4.2
 + cuda-bindings==13.3.1
 + cuda-pathfinder==1.6.1
 + cuda-toolkit==13.0.3.0
 + filelock==3.32.3
 + fsspec==2026.7.0
 + h11==0.16.0
 + hf-xet==1.6.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.28.0
 + idna==3.19
 + jinja2==3.1.6
 + joblib==1.5.3
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 + mdurl==0.1.2
 + mpmath==1.3.0
 + narwhals==2.25.0
 + networkx==3.6.1
 + numpy==2.5.2
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 + nvidia-cudnn-cu13==9.20.0.48
 + nvidia-cufft==12.0.0.61
 + nvidia-cufile==1.15.1.6
 + nvidia-curand==10.4.0.35
 + nv

## 4. Build llama.cpp with CUDA

The project's releases ship no prebuilt Linux CUDA binary, so the server is built
from source. `CMAKE_CUDA_ARCHITECTURES=native` compiles for this runtime's GPU
only, which cuts the build from tens of minutes to a few.

In [8]:
!apt-get -qq update
!apt-get -qq install -y cmake ninja-build

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [9]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp

fatal: destination path '/content/llama.cpp' already exists and is not an empty directory.


In [10]:
!cmake -S /content/llama.cpp \
  -B /content/llama.cpp/build \
  -G Ninja \
  -DCMAKE_BUILD_TYPE=Release \
  -DGGML_CUDA=ON \
  -DCMAKE_CUDA_ARCHITECTURES=native \
  -DLLAMA_CURL=OFF

-- llama.cpp version: 0.2.0-dev
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- CUDA Toolkit found
-- Using CMAKE_CUDA_ARCHITECTURES=75-real CMAKE_CUDA_ARCHITECTURES_NATIVE=75-real
-- CUDA host compiler is GNU 11.4.0
-- Including CUDA backend
-- ggml version: 0.21.0
-- ggml commit:  a130532
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (0.3s)
-- Generating done (0.2s)
-- Build files have been written to: /content/llama.cpp/build


In [11]:
!cmake --build /content/llama.cpp/build \
  --target llama-server \
  -j2

[263/318] Provisioning UI assets
-- UI: running npm ci

added 1067 packages, and audited 1068 packages in 41s

363 packages are looking for funding
  run `npm fund` for details

2 vulnerabilities (1 moderate, 1 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues, run:
  npm audit fix --force

Run `npm audit` for details.
-- UI: running npm run build, output -> /content/llama.cpp/build/tools/ui/dist

> llama-ui@1.0.0 build
> npm run build-pwa-assets && vite build


> llama-ui@1.0.0 build-pwa-assets
> npx @vite-pwa/assets-generator --root . --config pwa-assets.config.ts && npx @vite-pwa/assets-generator --root . --config pwa-assets-dark.config.ts && node scripts/make-icons-circular.js

Zero Config PWA Assets Generator v1.0.2
◐ Preparing to generate PWA assets...
◐ Resolving instructions...
✔ PWA assets ready to be generated, instructions resolved
◐ Generating PWA assets from static/favicon.svg image
◐ Generating assets for static/favicon.sv

In [13]:
!/content/llama.cpp/build/bin/llama-server --version

version: 0.2.0-dev (build 1, commit a130532)
built with GNU 11.4.0 for Linux x86_64


## 5. Download the quantised model (~2.3 GB)

In [14]:
!mkdir -p models
!curl -L --progress-bar -o models/Qwen3-4B-Instruct-2507-Q4_K_M.gguf \
    https://huggingface.co/unsloth/Qwen3-4B-Instruct-2507-GGUF/resolve/main/Qwen3-4B-Instruct-2507-Q4_K_M.gguf
!ls -lh models/

######################################################################## 100.0%
total 2.4G
-rw-r--r-- 1 root root 2.4G Aug 24 08:33 Qwen3-4B-Instruct-2507-Q4_K_M.gguf


## 6. Build the vector index — on CPU

`embed.py` pins the encoder to CPU. The index is 21 vectors built once to a
`.npy`, and at query time only the question is encoded, which is imperceptible
off-GPU. Watch the VRAM reading in the next cell: this step contributes nothing
to it, which is what frees the whole budget for the LLM.

In [15]:
!uv run python src/embed.py

encoding 21 chunks with BAAI/bge-m3 on CPU

















  emb_text.npy  (21, 1024)
  emb_value.npy  (21, 1024)
wrote /content/repo/data/index/meta.json — dim 1024, 21 chunks


## 7. Start llama-server and measure VRAM

`baseline` is what the GPU holds before the server starts, so the difference is
attributable to this workload rather than to whatever else the runtime is doing.

In [22]:
import subprocess, time, requests

MODEL = "models/Qwen3-4B-Instruct-2507-Q4_K_M.gguf"

def gpu_used_mib() -> int:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    return int(out.stdout.split()[0])

baseline = gpu_used_mib()

server = subprocess.Popen(
    ["/content/llama.cpp/build/bin/llama-server",
     "-m", MODEL, "--host", "127.0.0.1", "--port", "8081",
     "--ctx-size", "4096", "--cache-type-k", "q8_0", "--cache-type-v", "q8_0",
     "-ngl", "99", "--jinja"],
    stdout=open("/content/server.log", "wb"), stderr=subprocess.STDOUT,
)

for _ in range(180):
    try:
        if requests.get("http://127.0.0.1:8081/health", timeout=2).json().get("status") == "ok":
            break
    except Exception:
        pass
    time.sleep(1)

loaded = gpu_used_mib()
print(f"baseline before server : {baseline:>6} MiB")
print(f"after model load       : {loaded:>6} MiB   (+{loaded - baseline} MiB)")

baseline before server :      0 MiB
after model load       :   2887 MiB   (+2887 MiB)


## 8. VRAM under load

The KV cache grows with the tokens actually processed, so the honest figure is
taken after real traffic, not straight after loading. This cell fills the context
with the longest prompt the pipeline produces and re-reads the meter.

In [23]:
# Run through `uv run`, not this kernel: the project's dependencies live in the
# uv venv, and importing src/ here would need them installed a second time.
!uv run python src/rag.py "這台的顯卡、螢幕、連接埠和記憶體規格分別是什麼？" --max-tokens 320 --sources --base-url http://127.0.0.1:8081

under_load = gpu_used_mib()
print(f"\nunder load             : {under_load:>6} MiB   (+{under_load - baseline} MiB over baseline)")
print()
print(subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"],
    capture_output=True, text=True).stdout)
print(f"4096 MiB budget: {'WITHIN' if under_load - baseline < 4096 else 'EXCEEDED'}")


這台筆電的顯卡、螢幕、連接埠和記憶體規格如下：顯卡依機型而異，BZH 機型搭載 NVIDIA GeForce RTX 5090 Laptop GPU，BYH 機型搭載 NVIDIA GeForce RTX 5080 Laptop GPU，BXH 機型則搭載 NVIDIA GeForce RTX 5070 Ti Laptop GPU；螢幕為 16 吋 16:10 的 OLED WQXGA 面板，解析度為 2560×1600，更新率 240Hz，反應時間 1ms，並支援 NVIDIA G-SYNC 與 Dolby Vision；連接埠方面，左側提供 1 輸入電源插孔、1 個 RJ-45 以太網路埠、1 個 HDMI 2.1 埠、1 個支援 USB3.2 Gen2 的 Type-A 埠，以及 1 個支援 USB4、DisplayPort 2.1 與 Power Delivery 3.0 的 Type-C 埠（Thunderbolt 5）；右側則提供 1 個支援 USB4、DisplayPort 1.4 與 Power Delivery 3.0 的 Type-C 埠（Thunderbolt 4）、1 個 MicroSD（UHS-II）插槽，以及 1

TTFT 727 ms | 57.8 tok/s | 320 tokens | prompt 1028
sources: derived.overview, spec.memory, spec.ports

under load             :   2891 MiB   (+2891 MiB over baseline)

pid, process_name, used_gpu_memory [MiB]
32281, /content/llama.cpp/build/bin/llama-server, 2888 MiB

4096 MiB budget: WITHIN


## 9. Retrieval evaluation

No GPU involved — this scores the hybrid retriever against the 30-question golden
set and should reproduce the development numbers exactly, since nothing here is
sampled.

In [24]:
!uv run python eval/run_eval.py 2>/dev/null

30 questions: 23 answerable, 7 unanswerable (excluded from retrieval metrics), 8 alias-dependent

configuration               R@1    R@3  Hit@3    MRR   alias R@1
----------------------------------------------------------------
dense only, alias         0.471  0.986  1.000  0.826       0.250
bm25 only, alias          0.674  0.891  0.913  0.891       0.750
hybrid, no alias          0.304  0.768  0.826  0.616       0.250
hybrid + alias            0.645  1.000  1.000  0.928       0.750
  ...no SKU routing       0.609  0.986  1.000  0.884       0.750
  ...no overview chunk    0.645  1.000  1.000  0.935       0.750
(R@1 ceiling)             0.775      -      -      -       1.000

by question type — hybrid + alias        R@1 (ceiling)     R@3     MRR
  single_field     n=12       0.750 (1.000)   1.000   0.861
  cross_field      n=6        0.500 (0.500)   1.000   1.000
  model_diff       n=5        0.567 (0.567)   1.000   1.000

abstain signal (mean top-1 fusion score): answerable 0.1721  vs 

## 10. Generation evaluation

90 generations (30 questions × 3 configurations) at `temperature=0`. The TTFT and
TPS columns here are the ones the README reports; the Apple Silicon numbers are
kept only as a comparison.

In [25]:
!uv run python eval/run_gen_eval.py --base-url http://127.0.0.1:8081


config      facts   part  refuse  falseR  fab   簡  TTFT50  TTFT95    TPS
------------------------------------------------------------------------
no RAG      0.174  0.337   0.143   0.043    1   3   0.06s   0.08s   56.8
RAG k=1     0.696  0.819   0.857   0.435    0   1   0.08s   0.17s   52.7
RAG k=3     1.000  1.000   1.000   0.000    0   0   0.34s   0.49s   52.7

wrote /content/repo/eval/results/gen_eval.json


## 11. Final VRAM reading

In [26]:
peak = gpu_used_mib()
print(f"after the full evaluation: {peak} MiB used, {peak - baseline} MiB attributable to this workload")
!nvidia-smi

after the full evaluation: 2893 MiB used, 2893 MiB attributable to this workload
Mon Aug 24 09:14:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             43W /   70W |    2893MiB /  15360MiB |      0%      Default |
|                                         |                

## 12. Shut down

Frees the GPU so the reading above is the last word on what the workload held.

In [ ]:
server.terminate()
server.wait(timeout=30)
print("stopped")